In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/dhotepatil00@gmail.com/regis-healthcare/1_setup/utility

In [0]:
print(bronze_schema,silver_schema,gold_schema) 

In [0]:
dbutils.widgets.text("catalog","regis_healthcare","catalog")
dbutils.widgets.text("data_source","residents","data_source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

#### Silver Processing

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
display(df_bronze)
print(df_bronze.count())

In [0]:
# schema check
print(df_bronze.count())
df_bronze.printSchema()

In [0]:
df_bronze.columns

In [0]:
# drop duplicate
df_silver = df_bronze.dropDuplicates()
print(df_silver.count())

In [0]:
df_silver = df_silver.withColumn(
    "resident_id",
    F.trim(F.col("resident_id"))
).withColumn(
    "first_name",
    F.trim(F.col("first_name"))
).withColumn(
    "last_name",
    F.trim(F.col("last_name"))
).withColumn(
    "date_of_birth",
    F.trim(F.col("date_of_birth"))
).withColumn(
    "gender",
    F.trim(F.col("gender"))
).withColumn(
    "address",
    F.trim(F.col("address"))
).withColumn(
    "suburb",
    F.trim(F.col("suburb"))
).withColumn(
    "state",
    F.trim(F.col("state"))
).withColumn(
    "postcode",
    F.trim(F.col("postcode"))
).withColumn(
    "phone",
    F.trim(F.col("phone"))
).withColumn(
    "email",
    F.trim(F.col("email"))
).withColumn(
    "facility_id",
    F.trim(F.col("facility_id"))
).withColumn(
    "care_level",
    F.trim(F.col("care_level"))
).withColumn(
    "medicare_number",
    F.trim(F.col("medicare_number"))
).withColumn(
    "created_at",
    F.trim(F.col("created_at"))
).withColumn(
    "updated_at",
    F.trim(F.col("updated_at"))
)

In [0]:
# null records count 
from pyspark.sql.functions import col,count,when
null_count = df_silver.select([count(when(col(c).isNull(),c)).alias(c)for c in df_silver.columns
                               ])
display(null_count)

#### Cleaning data in table

In [0]:
# resident_id
from pyspark.sql.functions import col,when
filt = df_silver.filter(~col("resident_id").rlike("^//RES"))
display(filt)

In [0]:
# first_name
from pyspark.sql.functions import col,when,trim,initcap

df_silver = df_silver.withColumn("first_name",initcap(trim(col("first_name"))))

dup = df_silver.groupBy("first_name").count().filter(col("count")>1)
# dup.display()
df_invalid = df_silver.filter(col("first_name").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
# display(df_invalid)
from pyspark.sql import functions as F

df_silver = df_silver.withColumn(
    "first_name",
    F.regexp_replace("first_name", "@", "a")
)

df_silver = df_silver.withColumn(
    "first_name",
    F.regexp_replace("first_name", "3", "e")
)
df_invalid = df_silver.filter(col("first_name").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
display(df_invalid)
dup = df_invalid.groupBy("first_name").count()
display(dup)
display(df_silver)


In [0]:
# last_name
from pyspark.sql.functions import col,when,trim,initcap
df_silver = df_silver.withColumn("last_name",initcap(trim(col("last_name"))))

dup = df_silver.groupBy("last_name").count().filter(col("count")>1)
display(dup)
df_invalid = df_silver.filter(col("last_name").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
display(df_invalid)

In [0]:
# date_of_birth
from pyspark.sql.functions import col,when
from pyspark.sql import functions as F
df_silver = df_silver.withColumn(
    "date_of_birth",
    F.coalesce(
        # Date-only formats
        F.try_to_date(F.trim(F.col("date_of_birth")), F.lit("yyyy/MM/dd")),
        F.try_to_date(F.trim(F.col("date_of_birth")), F.lit("dd/MM/yyyy")),
        F.try_to_date(F.trim(F.col("date_of_birth")), F.lit("yyyy-MM-dd")),
        F.try_to_date(F.trim(F.col("date_of_birth")), F.lit("dd-MM-yyyy")),
        # Timestamp formats
        F.try_to_timestamp(F.trim(F.col("date_of_birth")), F.lit("yyyy-MM-dd HH:mm:ss")),
        F.try_to_timestamp(F.trim(F.col("date_of_birth")), F.lit("yyyy/MM/dd HH:mm:ss"))
    )
)
df_silver = df_silver.withColumn("date_of_birth", F.to_date("date_of_birth"))
display(df_silver)

In [0]:
# gender
from pyspark.sql.functions import col,when,upper,trim
df_silver = df_silver.withColumn("gender",upper(trim(col("gender"))))
dup = df_silver.groupBy("gender").count().filter(col("count")>1)
# display(dup)

fill_replace = {"FEMALE" : "FEMALE",
"MALE" : 'MALE',
"UNKNOWN" : 'NOT PROVIDE',
"F" : 'FEMALE',
"M" :'MALE',
"OTHER" : 'TRANS',
"X" : 'TRANS'
}
df_silver = (
    df_silver
    .withColumn("gender", F.upper(F.col("gender")))
    .replace(fill_replace, subset=["gender"]))
dup = df_silver.groupBy("gender").count().filter(col("count")>1)
display(dup)


In [0]:
# address
from pyspark.sql.functions import col,when,initcap,trim
df_silver = df_silver.withColumn("address",initcap(trim(col("address"))))
# df_invalid = df_silver.filter(col("address").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
# df_filt = df_silver.filter(col("address").rlike("^[0-9]+$"))
# df_filt = df_silver.filter(col("address").rlike("^[a-zA-Z]+$"))
df_filt = df_silver.filter(~col("address").rlike("[^a-zA-Z0-9]"))
display(df_filt)

In [0]:
# suburb
dup = df_silver.groupBy("suburb").count().filter(col("count")>1)
display(dup)

from pyspark.sql.functions import col,when,trim,upper
df_silver = df_silver.withColumn("suburb",upper(trim(col("suburb"))))
display(df_silver)

In [0]:
# state
from pyspark.sql.functions import col,when,trim,upper
dup = df_silver.groupBy("state").count()
# display(dup)

df_silver = df_silver.withColumn("state",upper(trim(col("state"))))
# display(df_silver)
dup = df_silver.groupBy("state").count()
display(dup)

In [0]:
# postcode
from pyspark.sql.functions import col,when,trim,upper
df_invalid = df_silver.filter(col("postcode").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
display(df_invalid)

In [0]:
# phone
from pyspark.sql.functions import col,when,trim,upper
df_silver = df_silver.withColumn("phone",when(col("phone").rlike("^[0-9]+$"),"Not Provide").otherwise(col("phone")))
df_filt = df_silver.filter(col("phone").rlike("^[0-9]+$"))
# display(df_filt)
dup = df_silver.groupBy("phone").count().filter(col("count")>1)
# display(dup)
# display(df_silver)

check = {"000-000-0000" : "Not Provide",
"ABCDEFG" : "Not Provide",
"+61 0" : "Not Provide",
"(02) 0000" : "Not Provide"}

df_silver = (
    df_silver.replace(check, subset=["phone"]))
dup = df_silver.groupBy("phone").count().filter(col("count")>1)
display(dup)
display(df_silver.limit(10))



In [0]:
# email
from pyspark.sql.functions import col,when,trim,upper
df_silver = df_silver.withColumn("email",trim(col("email")))
dup = df_silver.groupBy("email").count().filter(col("count")>1)
# display(dup)
# display(df_silver)

check = {"user@.com": "Not Provide",
"null" : "Not Provide",
"UNKNOWN": "Not Provide",
"user@" : "Not Provide",
"NONE": "Not Provide",
"user@@example.com" : "Not Provide",
"notanemail" : "Not Provide",
"NaN" : "Not Provide",
"null" : "Not Provide",
"@nodomain.com" : "mail@nodomain.com",
"NULL" : "Not Provide",
"N/A" : "Not Provide",
"" : "Not Provide",
"#N/A" : "Not Provide"}
df_silver = df_silver.fillna({"email" : "Not Provide"})
df_silver = (
    df_silver.replace(check, subset=["email"]))
dup = df_silver.groupBy("email").count().filter(col("count")>1)
display(dup)
dup = df_silver.filter(col("email")=="")
display(dup)

In [0]:
# facility_id
from pyspark.sql.functions import col,when
dev = df_silver.filter(~col("facility_id").rlike("^//FAC"))
display(dev)

In [0]:
# care_level
from pyspark.sql.functions import col,when,trim,upper
df_silver = df_silver.withColumn("care_level",upper(trim(col("care_level"))))
dup = df_silver.groupBy("care_level").count()
# display(dup)

repll = {"NULL" : "Not Provide",
          "N/A" : "Not Provide",
          ""   : "Not Provide",
          "NONE" : "Not Provide",
"#N/A" : "Not Provide",
"NAN" : "Not Provide",
"UNKNOWN" : "Not Provide",
"null" : "Not Provide"}

df_silver = (
    df_silver.replace(repll, subset=["care_level"]))
    
df_silver = df_silver.fillna({"care_level":"Not Provide"})
df_silver = df_silver.withColumn("care_level",upper(trim(col("care_level"))))
dup = df_silver.groupBy("care_level").count()
display(dup)



In [0]:
# medicare_number
from pyspark.sql.functions import col,when,trim,upper
df_silver = df_silver.withColumn("medicare_number",trim(col("medicare_number")))
dup = df_silver.groupBy("medicare_number").count().filter(col("count")>1)
# display(dup)

repll = {"N/A" : "NOT PROVIDE",
"UNKNOWN" : "NOT PROVIDE",
"NONE" : "NOT PROVIDE",
"null" : "NOT PROVIDE",
"NULL" : "NOT PROVIDE",
"NaN" : "NOT PROVIDE",
"null" : "NOT PROVIDE",
"" : "NOT PROVIDE",
"#N/A" : "NOT PROVIDE"
}

df_silver = (
    df_silver.replace(repll, subset=["medicare_number"]))
df_silver = df_silver.fillna({"medicare_number":"NOT PROVIDE"})
dup = df_silver.groupBy("medicare_number").count().filter(col("count")>1)
display(dup)

In [0]:
# created_at
df_invalid = df_silver.filter(~col("created_at").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
display(df_invalid)

In [0]:
# updated_at
df_invalid = df_silver.filter(~col("updated_at").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
display(df_invalid)

#### Silver table load

In [0]:
df_silver.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed","true")\
            .option("mergeSchema","true")\
                .option("overwriteSchema","true")\
            .mode("overwrite")\
               .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

dt = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source};")
print(dt.count())
display(dt)

In [0]:
# load to s3
df_silver.write.format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
    .mode("overwrite")\
    .partitionBy("current_date")\
    .save(f"s3://regis-healthcare/silver-clean-data/{data_source}/")